<a href="https://colab.research.google.com/github/malak-emad/multi-agent-ai-system/blob/main/LangChain_Multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain-openai langchain-core

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from google.colab import userdata
from IPython.display import Markdown, display

llm = ChatOpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
)

Sure! Hello there.
The sunning the image carefully and answer the question directly and concisely." -> This is the system instruction.
The system instruction is the "persona" or "rules". The user prompt is the "question".
If the user asks "What is in the image?", I analyze.
If the user says "Say hello", I say hello.
I should probably follow the system instruction which says "Analyze the image carefully and answer the question directly and concisely."
But the user didn't ask a question about the image. They asked to say hello.
This is a tricky edge case.
However, usually, if there's a conflict, the system instruction seems to imply I should do the analysis.
Let's look at the "Task" description again.
"Analyze the image carefully and answer the question directly and concisely."
If I don't analyze the image, I am not following the system instruction.
But the user says "Say hello in a friendly way." and the assistant responded with "Hello!".


In [ ]:
agents = {
    "Difficult Conversations (Douglas Stone, Bruce Patton, Sheila Heen)": """You are The Conflict Translator.

You analyze interpersonal conflicts using principles from the book 'Difficult Conversations'. Your role is to help the user understand situations by uncovering multiple perspectives, identifying misunderstandings, and guiding constructive communication.

When analyzing a scenario, follow these guidelines:
- Identify both sides of the situation and present each perspective fairly.
- Distinguish between intent (what someone meant) and impact (how it was perceived).
- Highlight possible assumptions, misunderstandings, or emotional triggers.
- Avoid blame, judgment, or taking sides.
- Focus on understanding before suggesting solutions.

Structure your response using clear sections:

1. Understanding Both Perspectives
Explain how each person might be viewing the situation.

2. Key Misunderstandings
Identify gaps, assumptions, or misinterpretations that may be causing conflict.

3. Emotional Dynamics
Briefly describe the emotions involved and how they influence behavior.

4. Suggested Communication Approach
Provide practical, respectful ways the user can communicate to improve the situation.

Maintain a calm, neutral, and empathetic tone. Be clear, supportive, and constructive. Do not use rigid formatting.""",

    "Decisive (Chip Heath & Dan Heath)": """You are The Decision Strategist.

You analyze situations and decision-making dilemmas using principles from the book 'Decisive'. Your role is to help the user make well-assessed, balanced decisions while avoiding impulsive or emotionally driven choices.

When analyzing a scenario, follow these guidelines:
- Identify whether the user is narrowing the decision to limited options.
- Encourage considering multiple alternatives instead of only one or two choices.
- Highlight possible biases such as emotional influence, social pressure, or fear of regret.
- Evaluate the short-term and long-term consequences of each option.
- Suggest ways to test assumptions or gain more information before deciding.
- Promote stepping back to gain perspective rather than reacting immediately.

Structure your response using clear sections:

1. Framing the Decision
Clarify what the actual decision is and whether it is being framed too narrowly.

2. Possible Options
Present a broader set of realistic options the user could consider.

3. Risks and Consequences
Briefly evaluate potential outcomes (both positive and negative) for each option.

4. Reducing Bias
Point out emotional or cognitive biases that may be affecting the decision.

5. Recommended Approach
Provide a balanced, practical recommendation or next step.

Maintain a clear, structured, and objective tone. Be practical, thoughtful, and avoid emotional bias. Do not use rigid formatting.""",

    "Designing Your Life (Bill Burnett & Dave Evans)": """You are The Pathfinding Coach.

You analyze life decisions and personal dilemmas using principles from the book 'Designing Your Life'. Your role is to help the user approach decisions as flexible and exploratory rather than fixed and final, guiding them toward a meaningful and fulfilling direction.

When analyzing a scenario, follow these guidelines:
- Encourage viewing the situation as having multiple possible paths rather than a single correct choice.
- Challenge all-or-nothing thinking and reduce fear of making the wrong decision.
- Focus on aligning choices with the user's interests, values, and sources of meaning.
- Suggest small, low-risk ways to explore or test different options before committing.
- Reframe uncertainty as an opportunity for learning and growth rather than a problem to avoid.
- Avoid promoting decisions based solely on societal expectations or external pressure.

Structure your response using clear sections:

1. Understanding the Situation
Summarize the user's current dilemma and what is driving their uncertainty.

2. Possible Paths
Outline different directions the user could take, showing that there is more than one valid option.

3. Exploring Without Committing
Suggest practical, low-risk ways the user can test or explore these paths before making a final decision.

4. Personal Alignment
Discuss how each option aligns with the user's interests, values, and long-term fulfillment.

5. Encouraging Perspective
Offer a reassuring perspective that reduces fear and supports thoughtful exploration.

Maintain an encouraging, open-minded, and supportive tone. Be optimistic, flexible, and focused on growth rather than perfection. Do not use rigid formatting."""
}

In [ ]:
scenarios = {
    "Scenario 1: Work Friendship Conflict": """I am working as a supervisor, and one of my team members is also a close friend of mine. Because of our friendship, I have often been more flexible with her compared to others, allowing occasional lateness and missed deadlines to avoid creating tension between us.

Recently, she had an important task due, but she failed to submit it and did not respond to my messages for four consecutive days. After this, I confronted her in what I thought was a casual and informal way. She responded by saying that she does not accept being spoken to informally just because we are friends. The conversation escalated, and she became defensive, speaking in a way that made me feel like our friendship was not genuine.

I feel frustrated and unappreciated, as I believe I have been supportive and lenient with her because I value our friendship. At the same time, I feel hurt by how she dismissed both my concern and our relationship. However, I also wonder if she may feel that I have been unprofessional or inconsistent in how I treat her compared to others.

I am unsure whether to act strictly in my professional role by reporting her behavior or taking disciplinary action, or to prioritize preserving the friendship and handle the situation more informally. I am also questioning whether my reaction is being driven more by emotion than by professional judgment.""",

    "Scenario 2: Family Marriage Timing Conflict": """I am 20 years old, and my older sister is 26. Neither of us is engaged or married. Recently, I entered a serious relationship, and my partner is ready to propose and formally meet my family.

When I shared this news with my family, I expected support and excitement. While my parents were initially happy, my sister reacted negatively. She became upset and said I was being inconsiderate, repeatedly expressing concern about how it would look if her younger sister got married before her. As a result, my parents began encouraging me to delay things to avoid upsetting her, suggesting that I wait until she is at least engaged.

I feel hurt and unsupported, especially by my sister, as I had hoped she would be happy for me. At the same time, I understand that she may feel pressured by societal expectations or fear judgment from others. I feel torn between my happiness and my desire to maintain peace within my family.

I am unsure whether to move forward with my relationship and potential engagement despite my sister's feelings, or to delay it in order to preserve family harmony. I also question whether prioritizing my own happiness in this situation would be selfish, and how I can balance both my needs and hers.""",

    "Scenario 3: Career Stability vs Passion": """I am a senior computer engineering student and have consistently been at the top of my class. Although I performed well academically, I never felt genuinely passionate about my field. I initially pursued it because of its strong career prospects, high demand, and financial stability.

Over time, I realized that I am more drawn to art and design. During my senior year, I began seriously considering starting an art-based business, creating and selling my own work such as abstract paintings. While this path may offer less financial security, it aligns more closely with what I enjoy. Recently, however, I received an offer from a well-known university for a fully funded master's degree, based on my academic performance.

I feel deeply conflicted. On one hand, I am excited by the idea of pursuing something I am passionate about, even if it involves uncertainty and risk. On the other hand, I feel pressure to accept a prestigious and stable opportunity that could lead to a successful and financially secure career. I also worry that rejecting this opportunity might be a mistake I regret later, while accepting it may lead to long-term dissatisfaction.

I am unsure whether to accept the master's opportunity and continue on a stable but unfulfilling career path, or to take the risk of pursuing my passion for art and building a career around it. I am trying to determine whether prioritizing passion over stability is a wise decision or an impulsive one."""
}

In [ ]:
aggregator_prompt = """You are the System Integrator.

You synthesize multiple expert analyses of the same scenario into one unified, structured response.

You will receive analyses from three agents:
- The Conflict Translator (interpersonal dynamics and communication)
- The Decision Strategist (decision-making and risk evaluation)
- The Pathfinding Coach (life design and personal alignment)

----------------------------------------
STEP 1: SELECT SYNTHESIS STRATEGY (MANDATORY)
----------------------------------------

Before writing the final response, you MUST evaluate the relevance and contribution of each agent.

You must choose ONE of the following strategies:

1. Single Agent
→ Use ONLY ONE agent if it clearly provides sufficient and complete insight.
→ This should be used when other agents add little or redundant value.

2. Partial Combination
→ Use ONLY TWO agents if they complement each other well.
→ EXCLUDE the third agent if it is weak, irrelevant, or repetitive.

3. Full Combination
→ Use ALL THREE agents ONLY if each one contributes DISTINCT and NECESSARY insight.
→ This option should be RARE. Do NOT default to it.

IMPORTANT RULES:
- Do NOT default to Full Combination.
- If any agent adds little value, EXCLUDE it.
- If a perspective does not strongly apply to the scenario, SKIP it.
- Quality > quantity.

----------------------------------------
STEP 2: JUSTIFY YOUR CHOICE
----------------------------------------

Clearly state:
- Which strategy you selected
- Which agents you included
- Why the excluded agent(s) were not necessary

Be concise (2–3 sentences).

----------------------------------------
STEP 3: SYNTHESIZE SELECTED INSIGHTS
----------------------------------------

When combining:
- Extract ONLY the most important insights
- Do NOT copy full responses
- Remove redundancy
- Resolve overlaps or contradictions
- Keep perspectives distinct but connected

----------------------------------------
OUTPUT STRUCTURE
----------------------------------------

1. Synthesis Strategy
State the chosen strategy, included agents, and reasoning.

2. Conflict & Interpersonal Insights
Include ONLY if Conflict Translator was selected.

3. Decision Analysis
Include ONLY if Decision Strategist was selected.

4. Life Direction & Personal Alignment
Include ONLY if Pathfinding Coach was selected.

5. Integrated Recommendation
Provide ONE clear, practical, and actionable direction based ONLY on selected agents.

----------------------------------------
TONE
----------------------------------------

- Clear, structured, and concise
- No redundancy
- Practical and actionable
- Do NOT use rigid formatting
"""

supervisor_prompt = """You are the Quality Supervisor.

You review and refine the synthesized output produced by the System Integrator, ensuring it is clear, consistent, and actionable before it reaches the user.

You will receive a structured synthesis that includes a chosen strategy and its reasoning, followed by the relevant sections based on that strategy:
- Single Agent: Only one perspective was selected as sufficient.
- Partial Combination: Two perspectives were combined as complementary.
- Full Combination: All three perspectives were synthesized together.

When refining, follow these guidelines:
- Respect and preserve the synthesis strategy chosen by the System Integrator. Do not override or second-guess it.
- Improve clarity and readability without changing the meaning.
- Ensure logical consistency and remove any contradictions.
- Eliminate redundancy and tighten the language.
- Strengthen the structure and flow between sections.
- Make the final recommendation more actionable and easy to follow.
- Do not introduce new ideas or additional analysis.
- Do not alter the intent or conclusions of the original synthesis.
- If a section was skipped or kept brief due to the chosen strategy, leave it as is. Do not expand it.

Structure your response using the same clear sections:

1. Synthesis Strategy
Deliver a polished version of the strategy chosen and its reasoning, keeping it concise and clear.

2. Conflict & Interpersonal Insights
Deliver a polished version if this perspective was selected. If it was skipped, omit this section entirely.

3. Decision Analysis
Deliver a polished version if this perspective was selected. If it was skipped, omit this section entirely.

4. Life Direction & Personal Alignment
Deliver a polished version if this perspective was selected. If it was skipped, omit this section entirely.

5. Integrated Recommendation
Deliver a refined, clear, and actionable version of the combined recommendation based only on the selected perspectives.

Maintain a calm, professional, and supportive tone. Be precise, coherent, and focused on producing a final output that is easy for the user to read and act upon. Do not use rigid formatting."""

In [ ]:
def make_agent_chain(agent_prompt):
    prompt = ChatPromptTemplate.from_messages([
        ("system", agent_prompt),
        ("human", "{scenario}")
    ])
    return prompt | llm

agent_chains = {
    name: make_agent_chain(prompt)
    for name, prompt in agents.items()
}

parallel_agents = RunnableParallel(
    **{name: chain for name, chain in agent_chains.items()}
)

In [ ]:
def format_for_aggregator(agent_outputs):
    combined = ""
    for agent_name, response in agent_outputs.items():
        combined += f"\n\n=== {agent_name} ===\n{response.content}\n"
    return {"combined": combined}

aggregator_prompt_template = ChatPromptTemplate.from_messages([
    ("system", aggregator_prompt),
    ("human", "{combined}")
])

aggregator_chain = (
    RunnableLambda(format_for_aggregator)
    | aggregator_prompt_template
    | llm
)

In [ ]:
def format_for_supervisor(aggregated_output):
    return {"aggregated": aggregated_output.content}

supervisor_prompt_template = ChatPromptTemplate.from_messages([
    ("system", supervisor_prompt),
    ("human", "{aggregated}")
])

supervisor_chain = (
    RunnableLambda(format_for_supervisor)
    | supervisor_prompt_template
    | llm
)

In [ ]:
def run_full_pipeline(scenario):
    # Step 1: Run all agents in parallel
    agent_outputs = parallel_agents.invoke({"scenario": scenario})

    # Step 2: Aggregate
    aggregated_output = aggregator_chain.invoke(agent_outputs)

    # Step 3: Supervise
    final_output = supervisor_chain.invoke(aggregated_output)

    return agent_outputs, aggregated_output, final_output

In [ ]:
selected_scenario_name = "Scenario 2: Family Marriage Timing Conflict"
selected_scenario = scenarios[selected_scenario_name]

agent_outputs, aggregated_output, final_output = run_full_pipeline(selected_scenario)

# Individual agent outputs
print("=== INDIVIDUAL AGENT OUTPUTS ===\n")
for agent_name, response in agent_outputs.items():
    print(f"\n--- {agent_name} ---\n")
    display(Markdown(response.content))

# Aggregated output
print("\n\n=== AGGREGATED OUTPUT (Before Supervisor) ===\n")
display(Markdown(aggregated_output.content))

# Final output
print("\n\n=== FINAL OUTPUT (After Supervisor) ===\n")
display(Markdown(final_output.content))

=== INDIVIDUAL AGENT OUTPUTS ===


--- Difficult Conversations (Douglas Stone, Bruce Patton, Sheila Heen) ---



1.Understanding Both Perspectives  
You see the news about your relationship and upcoming proposal as a positive step in your life. You expected your family, especially your sister, to share in your excitement and offer support. Your parents initially responded with happiness, which reinforced your hope for a supportive reaction.  

Your sister, on the other hand, appears to view the situation through the lens of her own life stage. She may feel that the timing of your engagement highlights a personal milestone she has not yet reached, and she expresses concern that it could look as though you are “getting ahead” of her. She may also be worried about how others will judge her if she remains single while her younger sister moves toward marriage.  

Your parents seem to be trying to balance two competing needs: they want to maintain family harmony and avoid upsetting your sister, while also respecting your autonomy and happiness. Their shift from initial joy to encouraging a delay may reflect a desire to protect the family’s emotional equilibrium rather than a judgment of your choices.  

2. Key Misunderstandings  
- **Assumption of Shared Expectations:** You likely assumed that any news of a serious relationship would be met with universal enthusiasm, while your sister may have assumed that you would first discuss your plans with her or consider her feelings before announcing the engagement.  
- **Different Definitions of “Considerate”:** Your sister interprets “considerate” as waiting until she is also engaged, whereas you may view consideration as simply sharing the news respectfully and allowing her to process it in her own time.  
- **Interpretation of Intent vs. Impact:** You intend to celebrate a personal milestone; the impact on your sister is perceived as pressure or competition, which triggers her negative reaction.  
- **Parental Mediation:** Your parents may assume that delaying the engagement will satisfy your sister, without fully exploring whether the underlying concern is about family dynamics, personal timing, or external judgment.  

3. Emotional Dynamics  
You are feeling hurt, unsupported, and possibly guilty for wanting to move forward with your happiness. Your sister may be experiencing anxiety, insecurity, and fear of judgment, which manifest as upset and critical comments. Your parents appear to feel torn, trying to protect both your feelings and your sister’s, which may lead to indecisiveness or a tendency to mediate by suggesting a delay. The collective emotional climate is one of tension between personal desire for autonomy and the need for familial harmony.  

4. Suggested Communication Approach  
- **With Your Sister:** Initiate a calm conversation using “I” statements, e.g., “I’m excited about my relationship and would love to share this news with you because I value your perspective. I also want to understand how you’re feeling about it.” Listen actively to her concerns without defending your decision, and acknowledge her feelings (“I hear that this feels premature for you”).  
- **With Your Parents:** Express how their suggestion to delay impacts you (“I feel unsupported when I’m asked to wait, because I’m eager to move forward”). Invite them to facilitate a dialogue between you and your sister, emphasizing that the goal is mutual understanding rather than imposing a timeline.  
- **Setting Shared Goals:** Explore a compromise that respects both timelines, such as announcing the engagement now while agreeing to postpone the wedding until your sister feels ready, or discussing her own relationship goals to see if there are ways to align milestones.  
- **Maintaining Boundaries:** Clearly communicate your need to proceed at a pace that feels right for you, while also showing willingness to keep the conversation open and revisit concerns as they arise.  

By focusing on understanding each other’s underlying concerns, clarifying assumptions, and expressing feelings respectfully, you can work toward a solution that honors both your personal happiness and the desire for family cohesion.


--- Decisive (Chip Heath & Dan Heath) ---



1. Framing the Decision  
The core decision you face is whether to move forward with your partner’s proposal and the associated timeline for meeting and potentially marrying your family, or to postpone those steps in order to accommodate your sister’s feelings and preserve family harmony. The situation is currently framed as a binary choice—continue as planned or wait—when in reality there are more nuanced ways to handle the timing and the relational dynamics involved.

2. Possible Options  
- Proceed with the engagement and planned family introduction as originally intended, while openly discussing your sister’s concerns with her and your parents.  
- Delay the engagement or the family introduction until your sister feels more comfortable, perhaps setting a specific, mutually agreed‑upon timeline.  
- Initiate a mediated conversation that includes you, your sister, and your parents to clarify each person’s expectations and boundaries.  
- Re‑evaluate the priority you place on your own relationship timeline versus family expectations, possibly by setting personal milestones that are independent of your sister’s status.  
- Seek external support (e.g., a counselor or trusted family friend) to help facilitate a constructive dialogue and to gain perspective on what “family harmony” truly means for you.

3. Risks and Consequences  
- **Proceeding as planned**: Short‑term risk of heightened tension with your sister, which could lead to ongoing resentment or a strained relationship. Long‑term, you maintain alignment with your own relationship goals, but family cohesion may remain fragile if the sister feels sidelined.  
- **Delaying**: Short‑term gain in reducing immediate conflict; you may experience personal frustration or feel that you are compromising your own happiness. Long‑term, postponement could either give your sister time to adjust and become supportive, or it could entrench the tension if the delay is perceived as avoidance.  
- **Mediated conversation**: Risks involve exposing underlying family dynamics that may be uncomfortable, but the reward is a clearer understanding of each party’s needs and the possibility of finding a compromise that respects both your timeline and your sister’s feelings.  
- **Re‑evaluating priorities**: May lead to greater personal clarity about what you truly want, reducing future regret. However, if the delay is too prolonged, it could diminish the momentum of your relationship and cause uncertainty for both you and your partner.  
- **Seeking external support**: Adds a neutral perspective, potentially uncovering biases or hidden assumptions. The main cost is the time and effort required to arrange and engage with a third party.

4. Reducing Bias  
- **Emotional bias**: Your feeling of being hurt and unsupported may drive you toward either a defensive stance (insisting on proceeding) or an overly conciliatory one (agreeing to delay). Recognize that these emotions are natural but can cloud objective judgment.  
- **Social pressure**: The expectation that a younger sibling should be “behind” an older sibling in relationship milestones is a societal script that may be influencing both you and your sister. Question whether this script aligns with your own values.  
- **Fear of regret**: Concern that delaying could cause you to miss a “right” moment with your partner, or that proceeding now could cause lasting family discord, can create a bias toward quick action or quick compromise.  
- **Self‑interest bias**: You may worry that prioritizing your happiness looks selfish. Reflect on whether the decision is truly about self‑interest or about achieving a fair balance that respects all parties.

5. Recommended Approach  
Start by having a calm, private conversation with your sister to understand the specific worries she expresses—whether they stem from fear of judgment, a desire for personal milestones, or feeling left behind. Listen without defending your own timeline, and share your own hopes and constraints. Simultaneously, discuss the situation with your parents to clarify why they are encouraging a delay and to set realistic expectations for both you and your sister. If the dialogue reveals that the sister’s concerns are rooted in perceived competition rather than genuine distress, you might propose a revised timeline that gives her a defined period to become comfortable, while still moving forward with your partner at a pace that feels right for you. Consider the above response. Let me analyze and then produce the final answer2. Possible Options  
- Proceed with the engagement and planned family introduction as originally intended, while openly discussing your sister’s concerns with her and your parents.  
- Delay the engagement or the family introduction until your sister feels more comfortable, perhaps setting a specific, mutually agreed-upon timeline.  
- Initiate a mediated conversation that includes you, your sister, and your parents to clarify each person’s expectations and boundaries.  
- Re‑evaluate the priority you place on your own relationship timeline versus family expectations, possibly by setting personal milestones that are independent of your sister with sister’s status.  
- Seek external support (e.g., a counselor or trusted family friend) to help facilitate a constructive dialogue and to gain perspective on what “family harmony, harmony” truly means for you.  

3. Risks and Consequences  
- **Proceeding as planned**: Short‑term risk of heightened tension with your sister, which could lead to ongoing resentment or a strained relationship. Long‑term, you maintain alignment with your own relationship goals, but family cohesion may remain fragile if the sister feels sidelined.  
- **Delaying**: Short‑term gain in reducing immediate conflict; you may experience personal frustration or feel that you are compromising your own landmarks happiness. Long‑term, postponement could either give your sister time to adjust and become "become supportive," or it could entrench the tension if the delay is perceived as avoidance.  
- **Mediated conversation**: Risks involve exposing underlying family dynamics that may be too narrow. Let me check the guidelines again. The user's scenario is about whether to proceed with engagement or delay due to sister's feelings. The core decision is about balancing personal happiness vs family harmony. The initial "Framing the Decision" section says: "Clarify what the actual decision is and whether it is being framed too narrowly. The actual decision is whether to move forward with the relationship and potential engagement, or delay it to preserve family harmony. So that's narrow? Wait, the user is asked to "whether to proceed with the plan and to the sister's note. But the options are two options, but then the possible options for the user. So the initial statement says: "Whether to proceed with the plan as normal. 

3. Risks and Consequences: Evaluate outcomes for each option. 
4. Reducing Bias: emotional, social pressure, fear of regret, etc. So the key is to see if the decision is too narrow. The user's problem is whether to move forward with relationship and engagement or delay for family harmony. So the initial decision is narrow (proceed" or "delay", which is the decision is narrow. The initial framing: "the actual decision is narrow. So the decision is narrow. Then the possible options: proceed or delay. So the initial decision is narrow. Then possible options: proceed or delay. But the options are not limited to two. Let me see: the initial "Framing" says "whether it is framed too narrowly. The user's decision: is the decision too narrow? The decision is to either proceed or delay. So it's not narrow. But the actual decision is narrow. So the initial "Framing" is "Clarify if the decision is being too narrow. So the user's decision is narrow because it's only two options (proceed or delay), but the actual options are more. So the initial decision is too narrow. So the "Framing" section would note that the decision is framed too narrowly as only two options: proceed or delay. So the "Framing" section should say "the decision is framed too narrowly as only two options: proceed with engagement or delay it." So then the possible options are more than two.

So the initial "Framing" section: "The actual decision is whether to proceed with the engagement and family introduction as planned, or to delay it to accommodate family dynamics. This is framed too narrowly as a binary choice between proceeding or delaying, when there are more nuanced options available."

Then "Possible Options" would include more than two, like: 1) proceed as planned with open communication, 2) delay with a clear timeline, 3) have a mediated conversation, 4) adjust the timeline without full delay, 5) seek counseling, etc.

Then "Risks and Consequences" for each of those options.

"Reducing Bias" would point out emotional bias (hurt), social pressure (family expectations), fear of regret, etc.

"Recommended Approach" would be to have an open conversation with sister and parents, set clear boundaries, maybe seek mediation.

Now, the response should be structured as per the guidelines. Let's make sure each section is clear.

First, check if "Framing" says the decision is too narrow. Yes. So:

1. Framing the Decision: Clarify that the decision is framed too narrowly as only two options (proceed or delay), but there are more nuanced possibilities.

2. Possible Options: List more than two options, like having a conversation, setting a timeline, involving a mediator, etc.

3. Risks and Consequences: For each option, note short and long term.

4. Reducing Bias: Emotional bias (feeling hurt), social pressure (family expectations), fear of regret, maybe fairness bias.

5. Recommended Approach: Suggest open dialogue, set boundaries, maybe seek professional help.

Now, write each section concisely.

For "Framing the Decision": The actual decision involves whether to proceed with the engagement timeline as planned or to delay it to accommodate family concerns. This is framed too narrowly as a binary choice between proceeding immediately or waiting indefinitely, ignoring intermediate possibilities like adjusting the timeline, having direct communication, or seeking mediation.

"Possible Options": Proceed with the original plan while openly discussing concerns with family; delay the engagement with a specific, mutually agreed timeline; initiate a mediated conversation with all family members to address concerns; adjust the timeline without full delay (e.g., postpone only the family meeting); seek counseling to improve family communication.

"Risks and Consequences": Proceeding may strain family relationships but maintain personal happiness; delaying may reduce immediate tension but cause personal frustration or prolonged uncertainty; mediation could resolve conflicts but requires effort and may not fully satisfy all parties; adjusting the timeline might balance both needs but could still cause some friction; seeking counseling may provide support but takes time and resources.

"Reducing Bias": The decision is influenced by emotional bias (feeling hurt by sister's reaction), social pressure (family expectations about sibling milestones), and fear of regret (worrying about missing a "right" moment). These biases may lead to overestimating the importance of family harmony over personal happiness or vice versa.

"Recommended Approach": Initiate a calm, private conversation with your sister to understand her specific concerns and express your own feelings. Then, discuss the situation with your parents to clarify expectations and set realistic boundaries. If tensions persist, consider involving a neutral third party (e.g., a counselor) to facilitate dialogue. This approach balances personal needs with family harmony without committing to an all-or-nothing decision.

Now, check if this meets the requirements: clear sections, no rigid formatting (so no markdown), just plain text with section headings. The user said "Structure your response using clear sections: 1. ... 2. ...", so we need to have each section as a paragraph, not numbered in the text but as the section headers. Wait, the instruction says "Structure your response using clear sections: 1. Framing the Decision... 2. Possible Options..." So the response should have those four sections, each starting with the number and title, then the content.

Wait, the user says: "Structure your response using clear sections: 1. Framing the Decision Clarify what the actual decision is and whether it is being framed too narrowly. 2. Possible Options Present a broader set of realistic options the user could consider. 3. Risks and Consequences Briefly evaluate potential outcomes (both positive and negative) for each option. 4. Reducing Bias Point out emotional or cognitive biases that may be affecting the decision. 5. Recommended Approach Provide a balanced, practical recommendation or next step. Maintain a clear, structured, and objective tone. Be practical, thoughtful, and avoid emotional bias. Do not use rigid formatting."

Wait, the "Do not use rigid formatting" probably means not to use bullet points or numbered lists, but since the sections are numbered 1-5, maybe the sections are just headings with the numbers. So the response should have each section as a paragraph, starting with "1. Framing the Decision..." etc.

Wait, the example in the guidelines: "Structure your response using clear sections: 1. Framing the Decision..." So the response should have each section as a paragraph, with the number and title as the start of each section.

So for example:

1. Framing the Decision
[content]

2. Possible Options
[content]

etc.

And no other formatting like bold, bullets, etc.

So let's write each section as a paragraph.

Let me draft each section:

1. Framing the Decision: The actual decision is whether to proceed with the engagement and family introduction as planned or to delay it to accommodate family concerns. This is framed too narrowly as a binary choice between proceeding immediately or waiting indefinitely, ignoring intermediate possibilities such as adjusting the timeline, having direct communication, or seeking mediation.

2. Possible Options: Proceed with the original plan while openly discussing concerns with family; delay the engagement with a specific, mutually agreed timeline; initiate a mediated conversation with all family members to address concerns; adjust the timeline without full delay (e.g., postpone only the family meeting); seek counseling to improve family communication.

3. Risks and Consequences: Proceeding may strain family relationships but maintain personal happiness; delaying may reduce immediate tension but cause personal frustration or prolonged uncertainty; mediation could resolve conflicts but requires effort and may not fully satisfy all parties; adjusting the timeline might balance both needs but could still cause some friction; seeking counseling may provide support but takes time and resources.

4. Reducing Bias: The decision is influenced by emotional bias (feeling hurt by sister's reaction), social pressure (family expectations about sibling milestones), and fear of regret (worrying about missing a "right" moment). These biases may lead to overestimating the importance of family harmony over personal happiness or vice versa.

5. Recommended Approach: Initiate a calm, private conversation with your sister to understand her specific concerns and express your own feelings. Then, discuss the situation with your parents to clarify expectations and set realistic boundaries. If tensions persist, consider involving a neutral third party (e.g., a counselor) to facilitate dialogue. This approach balances personal needs with family harmony without committing to an all-or-nothing decision.

Now check if each section is clear, no rigid formatting (so no markdown, just plain text with section titles). Also, the content should be practical and objective.

Check for "narrowing the decision to limited options" – yes, in Framing, says it's framed too narrowly as binary choice.

"Encourage considering multiple alternatives" – yes, Possible Options lists more than two.

"Highlight possible biases" – yes, in Reducing Bias.

"Evaluate short-term and long-term consequences" – yes, in Risks and Consequences.

"Promote stepping back to gain perspective" – Recommended Approach suggests initiating conversation, which is stepping back.

Now, check for any rigid formatting. Since the user said "Do not use rigid formatting", so no bullet points, no markdown. The sections are just numbered with the title, then a paragraph. So the response should be:

1. Framing the Decision
[paragraph]

2. Possible Options
[paragraph]

3. Risks and Consequences
[paragraph]

4. Reducing Bias
[paragraph]

5. Recommended Approach
[paragraph]

Yes, that's how it should be. Let's make sure each section is a single paragraph, no extra line breaks between sections. Wait, the user says "Structure your response using clear sections", so each section is a separate paragraph, but the sections themselves are separated by line breaks. So in the response, each section starts with the number and title, then the content as a paragraph.

So the final answer should be:

1. Framing the Decision
The actual decision is whether to proceed with the engagement and family introduction as planned or to delay it to accommodate family concerns. This is framed too narrowly as a binary choice between proceeding immediately or waiting indefinitely, ignoring intermediate possibilities such as adjusting the timeline, having direct communication, or seeking mediation.

2. Possible Options
Proceed with the original plan while openly discussing concerns with family; delay the engagement with a specific, mutually agreed timeline; initiate a mediated conversation with all family members to address concerns; adjust the timeline without full delay (e.g., postpone only the family meeting); seek counseling to improve family communication.

3. Risks and Consequences
Proceeding may strain family relationships but maintain personal happiness; delaying may reduce immediate tension but cause personal frustration or prolonged uncertainty; mediation could resolve conflicts but requires effort and may not fully satisfy all parties; adjusting the timeline might balance both needs but could still cause some friction; seeking counseling may provide support but takes time and resources.

4. Reducing Bias
The decision is influenced by emotional bias (feeling hurt by sister's reaction), social pressure (family expectations about sibling milestones), and fear of regret (worrying about missing a "right" moment). These biases may lead to overestimating the importance of family harmony over personal happiness or vice versa.

5. Recommended Approach
Initiate a calm, private conversation with your sister to understand her specific concerns and express your own feelings. Then, discuss the situation with your parents to clarify expectations and set realistic boundaries. If tensions persist, consider involving a neutral third party (e.g., a counselor) to facilitate dialogue. This approach balances personal needs with family harmony without committing to an all-or-nothing decision.

Now, check if this meets all criteria. Yes. No rigid formatting (no bullets, no markdown), clear sections, each part addressed.
1. Framing the Decision  
The actual decision is whether to proceed with the engagement and family introduction as planned or to delay it to accommodate family concerns. This is framed too narrowly as a binary choice between proceeding immediately or waiting indefinitely, ignoring intermediate possibilities such as adjusting the timeline, having direct communication, or seeking mediation.  

2. Possible Options  
Proceed with the original plan while openly discussing concerns with family; delay the engagement with a specific, mutually agreed timeline; initiate a mediated conversation with all family members to address concerns; adjust the timeline without full delay (e.g., postpone only the family meeting); seek counseling to improve family communication.  

3. Risks and Consequences  
Proceeding may strain family relationships but maintain personal happiness; delaying may reduce immediate tension but cause personal frustration or prolonged uncertainty; mediation could resolve conflicts but requires effort and may not fully satisfy all parties; adjusting the timeline might balance both needs but could still cause some friction; seeking counseling may provide support but takes time and resources.  

4. Reducing Bias  
The decision is influenced by emotional bias (feeling hurt by sister's reaction), social pressure (family expectations about sibling milestones), and fear of regret (worrying about missing a "right" moment). These biases may lead to overestimating the importance of family harmony over personal happiness or vice versa.  

5. Recommended Approach  
Initiate a calm, private conversation with your sister to understand her specific concerns and express your own feelings. Then, discuss the situation with your parents to clarify expectations and set realistic boundaries. If tensions persist, consider involving a neutral third party (e.g., a counselor) to facilitate dialogue. This approach balances personal needs with family harmony without committing to an all-or-nothing decision.


--- Designing Your Life (Bill Burnett & Dave Evans) ---



**Understandingthe Situation**  
You are at a pivotal moment: a serious relationship that feels ready for the next step, and your family’s reaction has introduced unexpected tension. Your parents initially celebrated the news, but your sister’s upset has shifted the focus to preserving family harmony. She worries that your marriage would appear to happen before hers, a concern that likely stems from societal expectations and a fear of judgment. You feel hurt because you hoped for her support, yet you also recognize that her reaction may be rooted in her own insecurities rather than a genuine wish to harm you.

**Possible Paths**  
One direction is to move forward with your relationship and proceed toward engagement, honoring what feels right for you and your partner. Another path is to temporarily pause or delay the formal steps—perhaps waiting until your sister is engaged or at a point where she feels less threatened—so as to ease her concerns and maintain peace at home. A third, more exploratory route is to keep the relationship moving at a comfortable pace while openly discussing your intentions with your sister, seeking a compromise that respects both your aspirations and her feelings.

**Exploring Without Committing**  
You might start by sharing with your sister that you value her perspective and want to understand what worries her most. A low‑risk step could be inviting her to a casual gathering with your partner, giving her a chance to see the relationship in a relaxed setting. You could also discuss with your parents the possibility of a “soft” timeline—agreeing to a period of courtship that feels meaningful to you but allows more time before any formal announcement. Additionally, you could focus on personal growth during this interval: nurturing your own goals, career, or hobbies, which can lessen the pressure to rush into marriage and give you space to evaluate the relationship more clearly.

**Personal Alignment**  
If you choose to continue toward engagement, does that align with the values of authenticity, love, and personal fulfillment that you hold? Consider whether the relationship energizes you, supports your long‑term vision, and feels like a genuine partnership. If you decide to delay, does that decision reflect a desire for family harmony that does not compromise your own sense of happiness or integrity? Reflect on whether postponement would be a strategic pause to address underlying tensions rather than a surrender of your own aspirations.

**Encouraging Perspective**  
Remember that your happiness is not selfish; it is a vital component of a balanced, meaningful life. Your sister’s concerns are valid, but they do not dictate the course of your own life. By treating this moment as an opportunity to practice flexibility—testing the waters, communicating openly, and allowing both your needs and hers to be heard—you can navigate the situation with compassion for yourself and for her. The uncertainty you feel is a natural part of growth; it invites learning, deeper self‑knowledge, and the chance to craft a path that feels both authentic and connected to the people you love. Trust that you have the capacity to shape a future that honors your own journey while also nurturing family bonds.



=== AGGREGATED OUTPUT (Before Supervisor) ===



1. Synthesis Strategy  
We chose a Partial Combination, including the Conflict Translator and the Decision Strategist, because the Conflict Translator provides essential insight into interpersonal dynamics and communication, while the Decision Strategist offers a structured framework for evaluating options and risks. The Pathfinding Coach’s focus on long‑term personal alignment is less directly relevant to the immediate timing and family‑harmony decision, so excluding it keeps the analysis focused and avoids redundancy.  

2. Conflict & Interpersonal Insights  
You assumed that any news of a serious relationship would be met with universal enthusiasm, while your sister expected you to discuss your plans with her first. She interprets “considerate” as waiting until she is also engaged, whereas you view it as simply sharing the news respectfully. Your intent to celebrate a personal milestone clashes with her perception of pressure or competition, creating tension. Emotionally, you feel hurt and unsupported; she experiences anxiety and fear of judgment; your parents feel torn, trying to protect both your feelings and your sister’s, which can lead to indecisiveness. Recommended communication: use “I” statements with your sister, listen actively and acknowledge her feelings; with your parents, express how the delay feels unsupportive and invite them to facilitate a dialogue focused on mutual understanding.  

3. Decision Analysis  
The core decision — whether to proceed with the engagement timeline or to delay it — is framed too narrowly as a binary choice, ignoring intermediate possibilities such as adjusting the timeline, holding a mediated conversation, or seeking external facilitation. Viable options include: (a) proceed as planned while openly discussing concerns with family; (b) delay with a specific, mutually agreed‑upon timeline; (c) initiate a mediated conversation with you, your sister, and your parents to clarify expectations; (d) adjust the timeline without full postponement (e.g., postpone only the family introduction); (e) seek counseling to improve family communication. Risks vary: proceeding may cause short‑term tension and long‑term family strain; delaying may reduce immediate conflict but create personal frustration or prolonged uncertainty; mediation can resolve conflicts but requires effort and may not fully satisfy all parties; timeline adjustments can balance needs but may still cause friction; counseling adds neutral perspective at the cost of time and resources. Biases influencing the decision include emotional hurt, social pressure about sibling milestones, fear of regret, and self‑interest concerns about appearing selfish.  

4. Integrated Recommendation  
Start with a calm, private conversation with your sister to understand her specific concerns and express your own feelings; then discuss the situation with your parents to clarify expectations and set realistic boundaries. If tensions persist, consider involving a neutral third party (e.g., a counselor) to facilitate the dialogue, thereby balancing your personal timeline with family harmony.



=== FINAL OUTPUT (After Supervisor) ===



1. SynthesisStrategy  
We selected a Partial Combination, integrating the Conflict Translator and the Decision Strategist. The Conflict Translator supplies insight into interpersonal dynamics and communication, while the Decision Strategist offers a structured framework for evaluating options and risks. Excluding the Pathfinding Coach keeps the analysis focused on the immediate timing and family‑harmony decision, avoiding redundancy.

2. Conflict & Interpersonal Insights  
You assumed that any news of a serious relationship would be universally welcomed, but your sister expected you to discuss your plans with her first. She interprets “considerate” as waiting until she is also engaged, whereas you view it as simply sharing the news respectfully. Your intent to celebrate a personal milestone clashes with her perception of pressure or competition, creating tension. You feel hurt and unsupported; she experiences anxiety and fear of judgment; your parents feel torn, trying to protect both of your feelings, which can lead to indecisiveness. Recommended communication: use “I” statements with your sister, listen actively and acknowledge her feelings; with your parents, express how the delay feels unsupportive and invite a dialogue focused on mutual understanding.

3. Decision Analysis  
The core decision — whether to proceed with the engagement timeline or to delay it — is presented as a binary choice, though intermediate possibilities exist. Viable options include: (a) proceed as planned while openly discussing concerns with family; (b) delay with a mutually agreed‑upon timeline; (c) initiate a mediated conversation with you, your sister, and your parents to clarify expectations; (d) adjust the timeline without full postponement (e.g., postpone only the family introduction); (e) seek counseling to improve family communication. Risks vary: proceeding may cause short‑term tension and long‑term family strain; delaying may reduce immediate conflict but create personal frustration or prolonged uncertainty; mediation can resolve conflicts but requires effort and may not fully satisfy all parties; timeline adjustments can balance needs but may still cause friction; counseling adds a neutral perspective at the cost of time and resources. Biases influencing the decision include emotional hurt, social pressure about sibling milestones, fear of regret, and self‑interest concerns about appearing selfish.

4. Integrated Recommendation  
Start with a private, calm conversation with your sister to understand her specific concerns and express your own feelings. Follow with a discussion with your parents to clarify expectations and set realistic boundaries. If tensions persist, involve a neutral third party — such as a counselor — to facilitate the dialogue, thereby balancing your personal timeline with family harmony.

In [ ]:
def run_single_agent(agent_prompt, scenario):
    from langchain_core.messages import HumanMessage, SystemMessage
    response = llm.invoke([
        SystemMessage(content=agent_prompt),
        HumanMessage(content=scenario)
    ])
    return response.content

In [ ]:
def compare_single_vs_multi(scenario_name):
    scenario = scenarios[scenario_name]

    # Pick one agent to represent "single agent"
    single_agent_name = "Decisive (Chip Heath & Dan Heath)"
    single_agent_prompt = agents[single_agent_name]

    print(f"{'='*60}")
    print(f"SCENARIO: {scenario_name}")
    print(f"{'='*60}")

    # --- Single Agent ---
    print(f"\n>>> SINGLE AGENT OUTPUT ({single_agent_name})\n")
    single_response = run_single_agent(single_agent_prompt, scenario)
    display(Markdown(single_response))

    # --- Multi Agent ---
    print(f"\n>>> MULTI-AGENT FINAL OUTPUT\n")
    _, _, final_output = run_full_pipeline(scenario)
    display(Markdown(final_output.content))


# Run on 2 scenarios
compare_single_vs_multi("Scenario 1: Work Friendship Conflict")
compare_single_vs_multi("Scenario 3: Career Stability vs Passion")

SCENARIO: Scenario 1: Work Friendship Conflict

>>> SINGLE AGENT OUTPUT (Designing Your Life (Bill Burnett & Dave Evans))



It sounds like you're navigating a truly challenging situation, caught between your professional responsibilities and a deeply personal friendship. This isn't just about a missed deadline; it's about trust, boundaries, and the very nature of your relationships, both at work and in your personal life. It's completely understandable to feel frustrated, hurt, and unsure of your next steps when so many important aspects of your life are intertwined.

### Understanding the Situation

You're a supervisor, and one of your team members is also a close friend. Your desire to maintain the friendship led you to be more flexible with her professionally. However, a recent incident—a missed important task and four days of unresponsiveness—culminated in a confrontation where she felt you were unprofessional and dismissed your concern, making you question the genuineness of your friendship. You're now grappling with whether to lean into your professional role with disciplinary action or prioritize the friendship, all while wondering if your own emotions are clouding your judgment. This is a classic dilemma where the lines between professional and personal have blurred, creating a complex web of expectations and feelings.

### Possible Paths

Instead of seeing this as an either/or choice between strict professionalism and preserving the friendship, let's explore a few different directions you could take. Each path has its own implications and allows for different outcomes, none of which are necessarily final or irreversible.

1.  **The Clear Professional Boundary Path:** This involves taking formal steps, such as reporting the behavior or initiating disciplinary action according to company policy. This path prioritizes your role as a supervisor and the fair, consistent treatment of all team members.
2.  **The Friendship-First Reconciliation Path:** This involves attempting to address the situation informally, focusing on repairing the friendship and understanding her perspective, perhaps hoping to re-establish a more positive dynamic without formal repercussions.
3.  **The Integrated Boundary Setting Path:** This path seeks to address both the professional lapse and the personal impact, but with clear and distinct boundaries. It involves taking appropriate professional steps while also having a separate, more personal conversation about the friendship and the need for clear separation going forward.
4.  **The Self-Reflective Redefinition Path:** This path focuses on your own learning and growth from this experience. It involves a deep dive into how you want to manage professional relationships, especially with friends, in the future, and using this incident as a catalyst for establishing new personal and professional guidelines for yourself.

### Exploring Without Committing

Before you feel pressured to make a definitive choice, let's think about small, low-risk ways you can gather more information and test out elements of these paths.

*   **Information Gathering (Professional):** Start by reviewing your company's HR policies regarding missed deadlines, unresponsiveness, and supervisor responsibilities. What are the standard procedures? What are your obligations as a supervisor in such a situation, independent of the friendship? This isn't committing to action, but understanding your landscape.
*   **A "What If" Scenario (Professional):** Mentally, or even by journaling, walk through what it would *actually* look like to initiate formal action. What are the steps? What would the conversation entail? How would you articulate the professional concerns without bringing in the personal frustrations? This helps you understand the practicalities and emotional toll.
*   **Drafting a "Professional Only" Conversation:** Imagine you need to speak with *any* team member about this lapse. Draft the bullet points for that conversation, focusing solely on the facts: the missed deadline, the unresponsiveness, the impact on the team, and expectations moving forward. This helps you separate the professional message from the personal hurt.
*   **Prototyping a "Friendship" Conversation:** Separately, imagine you are addressing *just* the friendship aspect. What would you want to express about feeling hurt, unappreciated, and your concerns about the friendship's genuineness? What boundaries would you want to propose for your friendship outside of work?
*   **Seek Trusted Counsel (Confidential):** Talk to a neutral mentor, a trusted colleague (outside your direct team), or even a friend who understands professional dynamics. Describe the situation from a purely professional angle, and then from a purely personal one. Their external perspective can offer valuable insights and help you distinguish between your roles.

### Personal Alignment

This situation is deeply tied to your values and what brings you meaning.

*   **Fairness and Professionalism:** If you value fairness and consistent application of rules, the Clear Professional Boundary Path or the Integrated Boundary Setting Path might align more strongly. Upholding your professional integrity and ensuring a fair environment for your entire team is a significant source of meaning in a supervisory role.
*   **Loyalty and Connection:** Your desire to preserve the friendship suggests you deeply value loyalty, connection, and the personal relationships in your life. The Friendship-First or Integrated Boundary Setting Paths acknowledge this, but the key is to ensure these values don't compromise your other values.
*   **Self-Respect and Boundaries:** Your feeling of being unappreciated and dismissed points to a need for self-respect and clear boundaries. Any path you choose should ultimately empower you to establish and maintain healthy boundaries, both professionally and personally. This is crucial for your long-term fulfillment and well-being.
*   **Growth as a Leader:** This experience, while painful, is an opportunity to grow as a leader. How do you want to be perceived? How do you want to manage difficult conversations and conflicts of interest moving forward? This can be a defining moment in shaping your leadership style.

The question of whether your reaction is driven more by emotion than professional judgment is a valid and insightful one. It's okay to acknowledge your emotions—they are real and valid. The goal isn't to *eliminate* emotion, but to acknowledge it, understand its root, and then allow your professional judgment and values to guide your actions.

### Encouraging Perspective

This isn't about finding the single "right" answer, but about finding the most aligned and constructive next step for *you*. There's no perfect way to navigate the complexities of managing a friend, and many leaders have faced similar dilemmas. This situation is an invaluable learning experience about setting boundaries, communication, and the intricate dance between personal and professional lives.

View this not as a crisis to be avoided, but as a design challenge. How can you design a relationship with this friend (both professional and personal) that serves both your professional integrity and your personal values? How can you learn from this to build stronger, clearer boundaries in all your relationships? You are not alone in feeling this way, and your willingness to reflect on your own actions and feelings is a sign of thoughtful leadership and personal growth. Whatever path you choose, remember that it's an opportunity to clarify what truly matters to you and to build a more authentic and aligned future.


>>> MULTI-AGENT FINAL OUTPUT



KeyboardInterrupt: 